# 📊 Social Sentiment Demand Tracker
## Predicting Retail Demand Direction from Social Media Signals

> *Part of a patent-pending AI retail system (USPTO Provisional — Patent Pending, June 5 2026)*

---

### What This Notebook Demonstrates

Traditional retail forecasting waits for sales data to confirm demand shifts. By then it's too late — shelves are empty or overflowing.

This notebook shows how **real-time social media sentiment** can predict demand direction **12–48 hours before** it appears in POS data — in **both directions**:

- 📈 **Positive signals** → accelerate replenishment before stockout
- 📉 **Negative signals** → suppress orders and flag markdowns before the sales decline

### Core Concepts

| Concept | Description |
|---|---|
| **SDDS** | Sentiment Demand Direction Score — signed [-1, +1] per post |
| **DDM** | Demand Direction Modifier — rolling exponentially-weighted SDDS |
| **Sentiment Velocity** | Rate of change of SDDS — early warning signal |


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"

from src.data_generator import generate_synthetic_posts
from src.sdds_engine import SDDSEngine
from src.ddm_engine import DDMEngine
from src.demand_signal import DemandSignalPipeline
from src.visualizer import (
    plot_sdds_timeseries, plot_ddm_comparison,
    plot_sentiment_heatmap, plot_post_volume, plot_gauge
)

print("✅ All imports successful")

## 2. Generate Synthetic Social Media Data

In [ ]:
# Generate 7 days of synthetic posts across 5 retail products
# Each product has a different scenario:
#   Stanley Tumbler    → viral positive (TikTok trend)
#   Crocs Classic      → plateauing trend
#   Ninja Blender      → complaint spiral (quality issues)
#   Oura Ring          → recall event (sharp negative shock)
#   Athletic Greens    → stable positive

raw_posts = generate_synthetic_posts(days=7, random_seed=42)

print(f"Total posts generated: {len(raw_posts):,}")
print(f"Date range: {raw_posts['timestamp'].min()} → {raw_posts['timestamp'].max()}")
print()
print(raw_posts.groupby(["product", "platform"]).size().unstack(fill_value=0))

In [ ]:
# Sample posts to see what synthetic data looks like
raw_posts[["timestamp", "product", "platform", "content",
           "ground_truth_sentiment", "likes"]].sample(10, random_state=1)

## 3. Compute SDDS — Sentiment Demand Direction Score

The **SDDS** is computed per post using VADER (Valence Aware Dictionary and sEntiment Reasoner):

- VADER is lexicon-based — fast, no GPU, works without training data
- Returns a compound score in **[-1, +1]**
- We weight by engagement (likes + 2× shares) so viral posts count more


In [ ]:
sdds_engine = SDDSEngine(volume_threshold=3)

# Score every post
scored_posts = sdds_engine.score_dataframe(raw_posts)

# Aggregate to hourly product-level SDDS
hourly_sdds = sdds_engine.compute_hourly_sdds(scored_posts)

print("Post-level SDDS statistics:")
print(scored_posts.groupby("product")["sdds"].agg(["mean", "std", "min", "max"]).round(3))
print()
print(f"Hourly observations: {len(hourly_sdds):,}")

In [ ]:
# Sentiment breakdown per product
print("=" * 60)
for product in scored_posts["product"].unique():
    b = sdds_engine.get_sentiment_breakdown(scored_posts, product)
    print(f"{product:<22} | +{b['positive']:>4} positive | "
          f"{b['neutral']:>4} neutral | -{b['negative']:>4} negative | "
          f"mean: {b['mean_sdds']:+.3f}")

## 4. Compute DDM — Demand Direction Modifier

The **DDM** is the operational signal. It's a rolling exponentially-weighted average of SDDS:

```
DDM_t = Σ (SDDS_τ × e^{-λ(t-τ)}) / Σ e^{-λ(t-τ)}
```

Key properties:
- **λ (decay rate)**: higher = more weight on recent signals
- **Volume threshold**: if fewer than N posts in window, SDDS = 0 (suppresses noise)
- **Sentiment Velocity**: slope of SDDS over last 4 hours — early warning


In [ ]:
ddm_engine = DDMEngine(decay_rate=0.15, lookback_hours=24, velocity_window=4)
ddm_series = ddm_engine.compute_ddm_series(hourly_sdds)

# Show latest signal per product
latest = ddm_engine.get_latest_signals(ddm_series)
print("LATEST OPERATIONAL SIGNALS")
print("=" * 80)
print(latest.to_string(index=False))

## 5. Visualise the Signals

In [ ]:
# DDM comparison across all products
fig = plot_ddm_comparison(latest)
fig.show()

In [ ]:
# Sentiment heatmap — products × time
fig = plot_sentiment_heatmap(hourly_sdds)
fig.show()

In [ ]:
# Post volume by platform over time
fig = plot_post_volume(scored_posts)
fig.show()

## 6. Product Deep-Dives

### 6a. Stanley Tumbler — Viral Positive Scenario

In [ ]:
fig = plot_sdds_timeseries(ddm_series, "Stanley Tumbler")
fig.show()

b = ddm_engine.get_latest_signals(ddm_series)
row = b[b["Product"] == "Stanley Tumbler"].iloc[0]
print(f"DDM: {row['DDM']:+.3f}  |  Action: {row['Recommended Action']}")

### 6b. Ninja Blender — Complaint Spiral Scenario
> *DDM declines steadily as complaints accumulate*

In [ ]:
fig = plot_sdds_timeseries(ddm_series, "Ninja Blender")
fig.show()

row = b[b["Product"] == "Ninja Blender"].iloc[0]
print(f"DDM: {row['DDM']:+.3f}  |  Action: {row['Recommended Action']}")

### 6c. Oura Ring — Recall Event Scenario
> *Sharp negative shock at t=40% then sustained decline*

In [ ]:
fig = plot_sdds_timeseries(ddm_series, "Oura Ring")
fig.show()

row = b[b["Product"] == "Oura Ring"].iloc[0]
print(f"DDM: {row['DDM']:+.3f}  |  Action: {row['Recommended Action']}")

## 7. The Inventory Action Engine

Based on the DDM value and Sentiment Velocity, the system assigns an inventory action:

| DDM Range | Velocity | Action |
|---|---|---|
| DDM ≥ +0.60 | Any | 🟢 Accelerate replenishment |
| DDM ≥ +0.25 | Any | 🟢 Standard replenishment |
| -0.30 ≤ DDM < +0.25 | Any | ⚪ Monitor |
| DDM < -0.30 | Any | 🟡 Suppress next order |
| DDM < -0.55 | Any | 🟠 Recommend markdown |
| DDM < -0.70 | Any | 🔴 Return-to-supplier alert |
| Any negative DDM | Velocity < -0.08 | ⚡ Early warning escalation |

The **velocity alert** is the most novel element — it fires when sentiment is accelerating downward even if the DDM hasn't crossed a threshold yet.


In [ ]:
# Show action distribution
action_counts = ddm_series["action"].value_counts()
print("Action distribution across all products × all hours:")
print(action_counts.to_string())

In [ ]:
# Find the exact hour when Oura Ring crossed into negative action territory
oura = ddm_series[ddm_series["product"] == "Oura Ring"].sort_values("hour")
first_negative = oura[oura["ddm"] < -0.30].iloc[0] if len(oura[oura["ddm"] < -0.30]) else None

if first_negative is not None:
    print(f"Oura Ring crossed DDM < -0.30 at: {first_negative['hour']}")
    print(f"  SDDS at that point: {first_negative['sdds_final']:+.3f}")
    print(f"  Velocity:           {first_negative['sentiment_velocity']:+.4f}")
    print(f"  Action triggered:   {first_negative['action']}")

## 8. Run the Full Pipeline (One-Liner)

In [ ]:
# The DemandSignalPipeline class wraps everything above
pipe = DemandSignalPipeline(days=7, decay_rate=0.15, lookback_hours=24).run()

# Access all intermediate outputs
print(f"\nPipeline outputs:")
print(f"  raw_posts:    {len(pipe.raw_posts):,} rows")
print(f"  scored_posts: {len(pipe.scored_posts):,} rows")
print(f"  hourly_sdds:  {len(pipe.hourly_sdds):,} rows")
print(f"  ddm_series:   {len(pipe.ddm_series):,} rows")

## 9. Connection to the Patent

This notebook demonstrates the **social media sentiment signal layer** of a larger patent-pending system:

> *"System and Method for Multi-Source Demand Signal Fusion and Real-Time Store-Level Inventory Forecasting Using Social Media Trend Analysis"*
> — USPTO Provisional Patent Application, June 5 2026

In the full system, the SDDS and DDM computed here are **two of seven input modalities** fused through a multi-modal transformer with cross-modal attention. The other modalities include:
- Proximity Intent Score (PIS) from "near me" search queries
- Search Intent Direction Score (SIDS) from product search query classification
- Hyperlocal weather signals
- Community event calendar signals
- Pay-cycle and academic calendar signals

The combined system generates a probabilistic (mean + variance) demand forecast at the **(SKU, store, hour)** granularity and triggers automated purchase orders or inventory reduction actions through an EDI-connected throttle controller.

---

*Follow the [AI in Retail newsletter](https://linkedin.com/in/gagansuri) for weekly posts on how AI is transforming retail.*
